# Policy Learning

- https://bookdown.org/stanfordgsbsilab/ml-ci-tutorial/policy-evaluation-i---binary-treatment.html
- https://bookdown.org/stanfordgsbsilab/ml-ci-tutorial/policy-learning-i---binary-treatment.html

이 둘을 파이썬으로 바꿔야하지 않을까 생각합니다.

causal ml 책에 있는 내용은 사실상 내용이 너무 적어서 굳이 볼 필요는 없는 것 같습니다. 우리가 파이썬 코드를 만들어야 합니다.

### 질문이나 의견을 남겨주세요.
<script src="https://utteranc.es/client.js"
        repo="CausalInferenceLab/awesome-causal-inference-python"
        issue-term="pathname"
        theme="github-light"
        crossorigin="anonymous"
        async>
</script>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import matplotlib.patches as mpatches

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate data
n = 1000
p = 4
X = np.random.uniform(0, 1, (n, p))
W = np.random.binomial(1, 0.5, n)  # Independent from X and Y
Y = 0.5 * (X[:, 0] - 0.5) + (X[:, 1] - 0.5) * W + 0.1 * np.random.randn(n)

In [ ]:
# Normalize Y for plotting
y_norm = 1 - (Y - Y.min()) / (Y.max() - Y.min())

# First plot: All data points
fig1, ax1 = plt.subplots(1, 1, figsize=(8, 6))
for i in range(n):
    if W[i] == 1:
        ax1.scatter(X[i, 0], X[i, 1], marker='o', s=100, 
                   c=[y_norm[i]], cmap='gray', vmin=0, vmax=1, 
                   edgecolors='black', linewidths=1)
    else:
        ax1.scatter(X[i, 0], X[i, 1], marker='D', s=80, 
                   c=[y_norm[i]], cmap='gray', vmin=0, vmax=1,
                   edgecolors='black', linewidths=1)
ax1.set_xlabel('X1', fontsize=12)
ax1.set_ylabel('X2', fontsize=12)
ax1.set_title('All Data Points (○: Treated, ◇: Untreated)', fontsize=14)
plt.show()

In [ ]:
# Second plot: Separated by treatment
fig2, (ax2, ax3) = plt.subplots(1, 2, figsize=(14, 6))

# Untreated group
untreated_idx = W == 0
ax2.scatter(X[untreated_idx, 0], X[untreated_idx, 1], marker='D', s=80, 
           c=y_norm[untreated_idx], cmap='gray', vmin=0, vmax=1,
           edgecolors='black', linewidths=1)
ax2.set_xlabel('X1', fontsize=12)
ax2.set_ylabel('X2', fontsize=12)
ax2.set_title('Untreated', fontsize=14)

# Treated group
treated_idx = W == 1
ax3.scatter(X[treated_idx, 0], X[treated_idx, 1], marker='o', s=100, 
           c=y_norm[treated_idx], cmap='gray', vmin=0, vmax=1,
           edgecolors='black', linewidths=1)
ax3.set_xlabel('X1', fontsize=12)
ax3.set_ylabel('X2', fontsize=12)
ax3.set_title('Treated', fontsize=14)
plt.show()

In [ ]:
# Third plot: Policy regions
fig3, ax4 = plt.subplots(1, 1, figsize=(8, 6))

# Define colors with transparency
col1 = (0.9960938, 0.7539062, 0.0273438, 0.35)  # Yellow-ish
col2 = (0.250980, 0.690196, 0.650980, 0.35)     # Teal-ish

# Draw policy regions
rect1 = Rectangle((-0.1, -0.1), 0.6, 1.2, linewidth=0, 
                  edgecolor='none', facecolor=col1, hatch='///')
rect2 = Rectangle((0.5, -0.1), 0.6, 0.6, linewidth=0, 
                  edgecolor='none', facecolor=col1, hatch='///')
rect3 = Rectangle((0.5, 0.5), 0.6, 0.6, linewidth=0, 
                  edgecolor='none', facecolor=col2, hatch='///')
ax4.add_patch(rect1)
ax4.add_patch(rect2)
ax4.add_patch(rect3)

# Plot data points
for i in range(n):
    if W[i] == 1:
        ax4.scatter(X[i, 0], X[i, 1], marker='o', s=100, 
                   c=[y_norm[i]], cmap='gray', vmin=0, vmax=1, 
                   edgecolors='black', linewidths=1)
    else:
        ax4.scatter(X[i, 0], X[i, 1], marker='D', s=80, 
                   c=[y_norm[i]], cmap='gray', vmin=0, vmax=1,
                   edgecolors='black', linewidths=1)

# Add text labels
ax4.text(0.75, 0.75, 'TREAT (A)', fontsize=16, ha='center', va='center')
ax4.text(0.25, 0.25, 'DO NOT TREAT (A^C)', fontsize=16, ha='left', va='center')
ax4.set_xlabel('X1', fontsize=12)
ax4.set_ylabel('X2', fontsize=12)
ax4.set_xlim(-0.1, 1.1)
ax4.set_ylim(-0.1, 1.1)
ax4.set_title('Policy Regions', fontsize=14)
plt.show()

In [ ]:
# Policy Evaluation Methods
print("=" * 60)
print("POLICY EVALUATION RESULTS")
print("=" * 60)

# Method 1: Value of policy A (only valid in randomized setting)
A = (X[:, 0] > 0.5) & (X[:, 1] > 0.5)
value_estimate = np.mean(Y[A & (W == 1)]) * np.mean(A) + \
                 np.mean(Y[~A & (W == 0)]) * np.mean(~A)
value_stderr = np.sqrt(
    np.var(Y[A & (W == 1)]) / np.sum(A & (W == 1)) * np.mean(A)**2 + 
    np.var(Y[~A & (W == 0)]) / np.sum(~A & (W == 0)) * np.mean(~A)**2
)
print(f"\nMethod 1: Value of Policy A")
print(f"Value estimate: {value_estimate:.6f}")
print(f"Std. Error: {value_stderr:.6f}")

In [ ]:
# Method 2: Value of fixed treatment proportion (p=0.75)
p_treat = 0.75
value_estimate2 = p_treat * np.mean(Y[W == 1]) + (1 - p_treat) * np.mean(Y[W == 0])
value_stderr2 = np.sqrt(
    np.var(Y[W == 1]) / np.sum(W == 1) * p_treat**2 + 
    np.var(Y[W == 0]) / np.sum(W == 0) * (1 - p_treat)**2
)
print(f"\nMethod 2: Value of Fixed Treatment Proportion (p={p_treat})")
print(f"Value estimate: {value_estimate2:.6f}")
print(f"Std. Error: {value_stderr2:.6f}")

In [ ]:
# Method 3: Treatment effect within policy region A
diff_estimate = (np.mean(Y[A & (W == 1)]) - np.mean(Y[A & (W == 0)])) * np.mean(A)
diff_stderr = np.sqrt(
    np.var(Y[A & (W == 1)]) / np.sum(A & (W == 1)) + 
    np.var(Y[A & (W == 0)]) / np.sum(A & (W == 0))
) * np.mean(A)
print(f"\nMethod 3: Treatment Effect within Policy Region A")
print(f"Difference estimate: {diff_estimate:.6f}")
print(f"Std. Error: {diff_stderr:.6f}")

In [ ]:
# Method 4: Optimal policy difference
diff_estimate2 = (np.mean(Y[A & (W == 1)]) - np.mean(Y[A & (W == 0)])) * np.mean(A) / 2 + \
                 (np.mean(Y[~A & (W == 0)]) - np.mean(Y[~A & (W == 1)])) * np.mean(~A) / 2
diff_stderr2 = np.sqrt(
    (np.mean(A) / 2)**2 * (
        np.var(Y[A & (W == 1)]) / np.sum(A & (W == 1)) + 
        np.var(Y[A & (W == 0)]) / np.sum(A & (W == 0))
    ) + 
    (np.mean(~A) / 2)**2 * (
        np.var(Y[~A & (W == 1)]) / np.sum(~A & (W == 1)) + 
        np.var(Y[~A & (W == 0)]) / np.sum(~A & (W == 0))
    )
)
print(f"\nMethod 4: Optimal Policy Difference")
print(f"Difference estimate: {diff_estimate2:.6f}")
print(f"Std. Error: {diff_stderr2:.6f}")

print("\n" + "=" * 60)

In [ ]:
# Additional analysis: Treatment effect heterogeneity
print("\nADDITIONAL ANALYSIS")
print("=" * 60)

# Calculate treatment effects by region
te_in_A = np.mean(Y[A & (W == 1)]) - np.mean(Y[A & (W == 0)])
te_out_A = np.mean(Y[~A & (W == 1)]) - np.mean(Y[~A & (W == 0)])

print(f"\nTreatment Effect Heterogeneity:")
print(f"Treatment effect in region A: {te_in_A:.6f}")
print(f"Treatment effect outside region A: {te_out_A:.6f}")
print(f"Difference in treatment effects: {te_in_A - te_out_A:.6f}")

In [ ]:
# Summary statistics
print(f"\nSummary Statistics:")
print(f"Proportion in region A: {np.mean(A):.3f}")
print(f"Proportion treated: {np.mean(W):.3f}")
print(f"Mean outcome (treated): {np.mean(Y[W == 1]):.6f}")
print(f"Mean outcome (untreated): {np.mean(Y[W == 0]):.6f}")
print(f"Overall treatment effect: {np.mean(Y[W == 1]) - np.mean(Y[W == 0]):.6f}")